In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict ,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [7]:
load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
class EssayTextState(TypeDict):
    essay:str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[lit[int], operator.add]
    avg_score: float
    

In [1]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description ='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10' ,ge= 0, le=10)
    

NameError: name 'BaseModel' is not defined

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

essay = """Artificial Intelligence (AI) is a modern technology that enables machines and computers to think, learn, and perform tasks like humans. It is becoming an important part of our daily lives.

AI is used in many fields, such as education, healthcare, business, transportation, and entertainment. In education, AI helps students learn new topics and solve problems. In healthcare, it helps doctors detect diseases and provide better treatment. AI-powered assistants, translation tools, and chatbots make our work easier and faster.

One of the biggest advantages of AI is that it can process large amounts of information quickly and accurately. It also helps save time, reduce human effort, and improve productivity. However, AI has some disadvantages too. Excessive dependence on AI may reduce human creativity and critical thinking. It can also create concerns about privacy, job loss, and the misuse of technology.

Therefore, AI should be used carefully and responsibly. Humans must ensure that AI supports society rather than causes harm. Proper rules and ethical practices are necessary to make AI safe and beneficial.

In conclusion, Artificial Intelligence is a powerful invention that can improve our lives in many ways. If used wisely, it can help build a smarter, safer, and better future for everyone.
"""

In [ ]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign the score out of 10 \n {essay}'

structured_model.invoke(prompt)
structured_model.invoke(prompt).feedback
structured_model.invoke(prompt).score

In [ ]:
 def evaluate_language(state:EssayTextState ):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign the score out of 10 \n {essay}'
    output = structured_model.invoke(prompt)

    return {'language_feedback':output.feedback, 'individual_scores':[output.score]}



In [ ]:
 def evaluate_analysis(state:EssayTextState ):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign the score out of 10 \n {essay}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback':output.feedback, 'individual_scores':[output.score]}



In [ ]:
 def evaluate_clarity(state:EssayTextState ):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign the score out of 10 \n {essay}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback':output.feedback, 'individual_scores':[output.score]}



In [ ]:

def final_evaluation(state: EssayTextState):

    # Summarize feedback from all evaluators
    prompt = f"""
    Based on the following feedback, create a summarized evaluation.

    Language feedback:
    {state["language_feedback"]}

    Depth of analysis feedback:
    {state["analysis_feedback"]}

    Clarity of thought feedback:
    {state["clarity_feedback"]}

    Provide an overall assessment with strengths and areas for improvement.
    """

    overall_feedback = model.invoke(prompt).content

    # Calculate average score
    scores = state["individual_scores"]
    avg_score = sum(scores) / len(scores) if scores else 0.0

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }


In [ ]:


graph = StateGraph(EssayTextState)

# Add nodes
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# Start three evaluations in parallel
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

# Join the three branches
graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")

# Finish
graph.add_edge("final_evaluation", END)

workflow = graph.compile()


In [ ]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)